# Build Dataset — GréineGrid RLThis notebook merges **Ausgrid** solar-home measurements with **AEMO** NSW wholesale prices into a single 30-minute dataset used by the RL environment.**Output:** `data/merged_30min_v2.csv`

## 1. Load Ausgrid solar home dataWe use the 2012–2013 Ausgrid file for customers 1–5. Each row is one consumption category (GC, GG, CL) with 48 half-hourly readings per day.

In [ ]:
import pandas as pdimport reausgrid = pd.read_csv(    "Ausgrid_solar_home_data/Solar home 2012-2013.csv",    skiprows=1,    parse_dates=["date"],)ausgrid = ausgrid[ausgrid["Customer"].isin([1, 2, 3, 4, 5])]ausgrid.head()

## 2. Reshape to long formatMelt the 48 time columns (`0:30`, `1:00`, …) into one row per customer × date × interval.

In [ ]:
hourly_regex = re.compile(r"^\d{1,2}:\d{2}$")hourly_columns = [col for col in ausgrid.columns if hourly_regex.match(col)]ausgrid = ausgrid.melt(    id_vars=["Customer", "Consumption Category", "date"],    value_vars=hourly_columns,    var_name="hourly",    value_name="kwh",)print(ausgrid.shape)print("Intervals per day:", ausgrid["hourly"].nunique())print("Categories:", ausgrid["Consumption Category"].unique())

### Column mapping| Ausgrid code | Meaning | Used as ||--------------|---------|----------|| **GC** | General consumption | `load_kwh` || **GG** | Gross solar generation | `pv_kwh` || **CL** | Controlled load (e.g. hot water) | Dropped — not part of the battery dispatch model |

In [ ]:
ausgrid_wide = ausgrid.pivot_table(    index=["Customer", "date", "hourly"],    columns="Consumption Category",    values="kwh",    aggfunc="sum",).reset_index()ausgrid_wide = (    ausgrid_wide    .rename(columns={"GC": "load_kwh", "GG": "pv_kwh"})    .drop(columns=["CL"]))ausgrid_wide.columns.name = Noneprint(ausgrid_wide.head())print(ausgrid_wide.isna().sum())

## 3. Build timestamps- Combine calendar date + interval label into a datetime.- The `0:00` reading belongs to the **next** calendar day (end-of-day interval).- Localise to `Australia/Sydney`, convert to UTC, and drop ambiguous DST rows.

In [ ]:
ausgrid_wide["timestamp"] = pd.to_datetime(    ausgrid_wide["date"] + " " + ausgrid_wide["hourly"],    dayfirst=True,)is_midnight = ausgrid_wide["hourly"] == "0:00"ausgrid_wide.loc[is_midnight, "timestamp"] += pd.Timedelta(days=1)ausgrid_wide["timestamp"] = (    ausgrid_wide["timestamp"]    .dt.tz_localize("Australia/Sydney", ambiguous="NaT", nonexistent="NaT")    .dt.tz_convert("UTC"))n_nat = ausgrid_wide["timestamp"].isna().sum()print(f"Rows dropped due to DST ambiguity: {n_nat}")ausgrid_wide = ausgrid_wide.dropna(subset=["timestamp"])

In [ ]:
# Sanity check: one customer-day should span 48 consecutive 30-min stepsday_one = ausgrid_wide[    (ausgrid_wide["Customer"] == 1) & (ausgrid_wide["date"] == "1/07/2012")]print(day_one.sort_values("timestamp")[["hourly", "timestamp"]].iloc[[0, -1]])

## 4. Load AEMO wholesale pricesNSW1 30-minute settlement prices (RRP in AUD/MWh → converted to AUD/kWh).

In [ ]:
import globprices = pd.concat(    [pd.read_csv(f) for f in glob.glob("aemo/*.csv")],    ignore_index=True,)prices["timestamp"] = (    pd.to_datetime(prices["SETTLEMENTDATE"])    .dt.tz_localize("Etc/GMT-10")    .dt.tz_convert("UTC"))prices.rename(columns={"TOTALDEMAND": "total_demand_mw"}, inplace=True)prices["price_per_kwh"] = prices["RRP"] / 1000print(prices.head())print(prices.shape)print(prices["price_per_kwh"].describe())

## 5. Merge Ausgrid + AEMO on timestampInner join keeps only intervals where both load/PV and price data exist.

In [ ]:
ausgrid_merge = pd.merge(    ausgrid_wide,    prices[["timestamp", "price_per_kwh", "total_demand_mw"]],    on="timestamp",    how="inner",)print("Before merge:", len(ausgrid_wide))print("After merge:", len(ausgrid_merge))print(ausgrid_merge[["price_per_kwh", "total_demand_mw"]].isna().sum())

## 6. Finalise columns and export`net_load_kwh = load_kwh - pv_kwh` (negative values indicate solar export surplus).

In [ ]:
ausgrid_merge["net_load_kwh"] = ausgrid_merge["load_kwh"] - ausgrid_merge["pv_kwh"]ausgrid_merge.rename(columns={"Customer": "customer_id"}, inplace=True)ausgrid_merge.sort_values(by=["customer_id", "timestamp"], inplace=True)final = ausgrid_merge[[    "customer_id",    "timestamp",    "pv_kwh",    "load_kwh",    "price_per_kwh",    "total_demand_mw",    "net_load_kwh",]].copy()# Store timestamps in Sydney local time for readabilityfinal["timestamp"] = final["timestamp"].dt.tz_convert("Australia/Sydney")final.to_csv("data/merged_30min_v2.csv", index=False)print(final.columns.tolist())print("Shape:", final.shape)print("Rows with net export (net_load < 0):", (final["net_load_kwh"] < 0).sum())final.head()

## 7. Quick validationEach RL episode is one calendar day with exactly 48 steps. Run `data_split.ipynb` next to filter complete days and create the train/val/test split.

In [ ]:
df = final.copy()df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert("Australia/Sydney")df["episode_day"] = (df["timestamp"] - pd.Timedelta(minutes=1)).dt.datesteps = df.groupby(["customer_id", "episode_day"]).size()incomplete = steps[steps != 48]print("Days per customer:")print(df.groupby("customer_id")["episode_day"].nunique())print()for customer in sorted(df["customer_id"].unique()):    n_bad = (incomplete.index.get_level_values("customer_id") == customer).sum()    print(f"Customer {customer}: {n_bad} incomplete days (not 48 steps)")

---**Next step:** open `data_split.ipynb` to drop incomplete days, exclude customer 2, and write `data/day_split.json`.